# modulos/deteccao — detecção automática de abas e colunas (multi-edição)

Módulo carregado pelo hub `environment/env_functions`. Com ele, uma edição nova da MUNIC roda **sem nenhuma configuração nova por ano** — os overrides por edição são um escape opcional (vazios no dia a dia) e ficam no `env_variables`.

## O problema

Os nomes das abas do XLSX e das colunas que identificam o município mudam entre as edições da MUNIC — às vezes por acento, às vezes até por typo do próprio IBGE:

| O que muda | Edição 2024 | Edição 2019 |
| --- | --- | --- |
| Aba de informática | `Informática e comunicação` | `Comunicção e informática` (typo) |
| Aba de governança | `Governanca` (sem cedilha) | `Governança` |
| Código do município | `Cod Munic` | `CodMun` |
| Nome do município | `Desc Mun` | `NOME MUNIC` |
| Porte populacional | `Faixa_populacao` | `CLASSE POP` |

Antes, cada diferença dessas virava uma linha em mapas por edição (`munic_sheet_map` / `munic_identity_rename_map`) — todo ano novo de pesquisa exigia configurar de novo.

## Fluxo (3 passos)

1. **Normalizar** — `_nome_canonico()` reduz qualquer nome a uma forma comparável: minúsculo, sem acentos e só letras/números. Assim `Governança`, `Governanca` e `GOVERNANÇA` viram `governanca`, e `Cod Munic` vira `codmunic`.
2. **Reconhecer** — abas são localizadas por marcadores com radicais e sinônimos (`informat`/`informac` + `comunic`, `governan`, `dicion`), tolerando renomeações como 'Tecnologia da informação' ou 'TI'; colunas de identidade são localizadas por regras simples ("começa com cod e contém mun" = código do município, e assim por diante).
3. **Falhar rápido** — se mais de uma aba ou coluna casar com o padrão, a função aborta com uma mensagem explicando o que fazer, em vez de ingerir a aba/coluna errada silenciosamente.

## Como usar (o landing_to_bronze já faz as duas chamadas)

    abas    = detect_munic_sheets(pd.ExcelFile(munic_file).sheet_names)
    renomes = detect_identity_renames(colunas_da_edicao)

**Detalhe:** coluna ausente numa edição (ex.: `populacao` não existe na base 2019) não gera renome — o alinhamento de schema do upsert da bronze preenche com null.

**Escape opcional:** se uma edição futura confundir a detecção (ex.: duas abas parecidas), registre um override em `munic_sheet_overrides` / `munic_identity_rename_overrides` no `env_variables`.

In [0]:
import re
import unicodedata


# --- Passo 1: a forma canônica de um nome -----------------------------------
# Toda comparação abaixo usa esta forma, imune a maiúsculas, acentos,
# espaços e separadores: 'Governança' == 'Governanca' == 'governanca'.
def _nome_canonico(nome):
    """Reduz um nome a minúsculo, sem acentos e apenas letras/números.

        'Cod Munic'                -> 'codmunic'
        'Comunicção e informática' -> 'comuniccaoeinformatica'
        'Governança'               -> 'governanca'
    """
    sem_acento = unicodedata.normalize("NFKD", str(nome))
    sem_acento = "".join(ch for ch in sem_acento if not unicodedata.combining(ch))
    return re.sub(r"[^a-z0-9]", "", sem_acento.lower())


# --- Passo 2a: como reconhecer cada ABA do XLSX ------------------------------
# Cada aba é identificada por MARCADORES alternativos compostos de radicais:
# a aba casa se TODOS os tokens de QUALQUER marcador estiverem no nome canônico.
ABAS_TOKENS = {
    # Usamos radicais + sinônimos para sobreviver a renomeações entre edições
    # (ex.: 'informática' -> 'tecnologia da informação' ou 'TI').
    # chave   marcadores alternativos (tuplas de tokens)
    "dict": (
        ("dicion",),                       # 'Dicionário', 'Dicionário de dados'...
    ),
    "info": (
        ("informat", "comunic"),           # 'Informática e comunicação' | 'Comunicção e informática' (typo 2019)
        ("informac", "comunic"),           # 'Tecnologia da informação e comunicação'
        ("tiecomun",),                     # 'TI e comunicação'
    ),
    "gov": (
        ("governan",),                     # 'Governança' | 'Governanca' (sem cedilha) | 'Governança municipal'
    ),
}

def detect_munic_sheets(sheet_names):
    """A partir dos nomes de abas do XLSX, resolve {'dict': aba, 'info': aba, 'gov': aba}.

    Uso (o landing_to_bronze já faz isso na célula de leitura das planilhas):
        abas = detect_munic_sheets(pd.ExcelFile(munic_file).sheet_names)
        # -> {'dict': 'Dicionário', 'info': 'Comunicção e informática', 'gov': 'Governança'}

    Se alguma aba não for achada com exclusividade, falha aqui (fail-fast)
    com instrução de como resolver — nunca ingere a aba errada silenciosamente.
    """
    canonicos = {aba: _nome_canonico(aba) for aba in sheet_names}
    resolvidas = {}
    for alvo, marcadores in ABAS_TOKENS.items():
        achadas = [
            aba for aba, nome in canonicos.items()
            if any(all(t in nome for t in tokens) for tokens in marcadores)
        ]
        assert len(achadas) == 1, (
            f"Não consegui resolver a aba '{alvo}' (procurei os marcadores {marcadores}): "
            f"achei {achadas} entre as abas {list(sheet_names)}. "
            f"Se a edição realmente divergir, registre um override em munic_sheet_overrides."
        )
        resolvidas[alvo] = achadas[0]
    return resolvidas


# --- Passo 2b: como reconhecer cada COLUNA de identidade ---------------------
# As colunas que identificam o município mudam de nome entre edições;
# as regras abaixo reconhecem cada uma pelo seu nome canônico.
def _eh_cod_municipio(nome):    # 'CodMun' (2019) | 'Cod Munic' (2024) | 'cod_munic' (já canônico)
    return nome.startswith("cod") and "mun" in nome

def _eh_nome_municipio(nome):   # 'NOME MUNIC' (2019) | 'Desc Mun' (2024) | 'desc_mun' (já canônico)
    return "mun" in nome and ("nome" in nome or "desc" in nome)

def _eh_faixa_populacao(nome):  # 'CLASSE POP' (2019) | 'Faixa_populacao' (2024)
    return any(pedaco in nome for pedaco in ("classe", "faixa", "porte"))

# (nome canônico na bronze, o que a coluna significa, regra de reconhecimento)
COLUNAS_IDENTIDADE = [
    ("cod_munic",       "código IBGE do município",   _eh_cod_municipio),
    ("desc_mun",        "nome do município",          _eh_nome_municipio),
    ("faixa_populacao", "faixa de porte populacional", _eh_faixa_populacao),
]

def detect_identity_renames(colunas):
    """A partir das colunas de uma edição, gera os renomes {original: canônico} para a bronze.

    Uso (o landing_to_bronze já faz isso antes do upsert):
        renomes = detect_identity_renames(['codmun', 'nome_munic', 'classe_pop', 'mtic011', ...])
        # -> {'codmun': 'cod_munic', 'nome_munic': 'desc_mun', 'classe_pop': 'faixa_populacao'}

    Observações:
    - Coluna já canônica (ex.: 2024 'Cod Munic' -> 'cod_munic') não gera renome.
    - Coluna ausente numa edição (ex.: 'populacao' não existe em 2019) também não gera renome:
      o upsert da bronze completa colunas faltantes com null (alinhamento de schema).
    """
    renomes = {}
    for alvo, _significado, reconhece in COLUNAS_IDENTIDADE:
        candidatas = [col for col in colunas if col != alvo and reconhece(_nome_canonico(col))]
        assert len(candidatas) <= 1, (
            f"A coluna canônica '{alvo}' ficou ambígua: candidatas {candidatas} entre {colunas}. "
            f"Registre um override em munic_identity_rename_overrides."
        )
        if candidatas:
            renomes[candidatas[0]] = alvo
    return renomes